In [1]:
import uuid
from IPython.display import display, HTML

def create_hide_button():
    unique_id = str(uuid.uuid4())
    display(HTML('''
        <script>
        function toggle_output(button_id) {
            var output_area = document.getElementById(button_id).parentElement.parentElement.nextElementSibling;
            var button = document.getElementById(button_id);
            if (output_area.style.display === "none") {
                output_area.style.display = "block";
                button.innerHTML = "Hide Output";
            } else {
                output_area.style.display = "none";
                button.innerHTML = "Show Output";
            }
        }
        </script>
    '''))

    button_id = f"button_{unique_id}"
    display(HTML(f'''
        <button id="{button_id}" onclick="toggle_output('{button_id}')">Show Output</button>
        <script>
            document.getElementById('{button_id}').parentElement.parentElement.nextElementSibling.style.display = 'none';
        </script>
    '''))

In [2]:
from IPython.display import display, HTML

display(HTML('''<style>
    .jp-CodeCell .jp-InputArea {display: none;} /* Hide only code cells */
</style>'''))

In [3]:
from pathlib import Path
import pandas as pd
import re
import seaborn as sns
import os
import math
from matplotlib.patches import Patch
import numpy as np
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
from pathlib import Path
import colorsys

from tigramite import data_processing as pp
from tigramite.independence_tests.parcorr import ParCorr
from tigramite.pcmci import PCMCI
from tigramite import plotting as tp

import warnings
warnings.filterwarnings('ignore')


In [4]:

plt.rcParams.update({
    'font.size': 14,              
    'axes.titlesize': 16,         
    'axes.labelsize': 16,         
    'xtick.labelsize': 14,        
    'ytick.labelsize': 14,        
    'legend.fontsize': 14,        
    'figure.titlesize': 18,       
    'grid.linewidth': 1.2         
})

In [5]:
current_path = Path(__file__).resolve().parent if '__file__' in globals() else Path().resolve()
folder = current_path.parent.parent / 'data' / 'silver'

# Prepare results

In [6]:

# Do a partial correlation for every file in silver

folder_path = folder


parquet_files = list(folder_path.rglob('*.parquet'))




results = []


for file_path in parquet_files:
    print(f"Processing {file_path.name}...")
    

    folder_name = file_path.parent.name
    
    df = pd.read_parquet(file_path)
    df_no_time = df.drop(columns=['time'])
    data = df_no_time.values
    var_names = df_no_time.columns.tolist()
    dataframe = pp.DataFrame(data, 
                             datatime={0: np.arange(len(data))}, 
                             var_names=var_names)
    
    # Compute correlations
    parcorr = ParCorr(significance='analytic')
    pcmci = PCMCI(
        dataframe=dataframe, 
        cond_ind_test=parcorr,
        verbosity=0)
    correlations = pcmci.get_lagged_dependencies(tau_max=20, tau_min=1,val_only=True)['val_matrix']
    
    # Compute matrix_lags
    matrix_lags = np.argmax(np.abs(correlations), axis=2)
    
    # Compute mean and median lag
    mean_lag = matrix_lags.mean()
    median_lag = np.median(matrix_lags)
    
    # Store the result
    dataset_name = file_path.stem 
    results.append({
        'folder': folder_name,
        'dataset': dataset_name,
        'median_lag': median_lag
    })
    
    print(f"Folder: {folder_name}, Dataset: {dataset_name}, Mean lag: {mean_lag}, Median lag: {median_lag}")

# Create dataframe from results
results_df = pd.DataFrame(results)

Processing df_rack_1.parquet...
Folder: dc1_prepared, Dataset: df_rack_1, Mean lag: 6.568047337278107, Median lag: 5.0
Processing df_rack_3.parquet...
Folder: dc1_prepared, Dataset: df_rack_3, Mean lag: 7.485207100591716, Median lag: 5.0
Processing df_rack_2.parquet...
Folder: dc1_prepared, Dataset: df_rack_2, Mean lag: 8.071005917159763, Median lag: 7.0
Processing df_5m.parquet...
Folder: dc2_2_prepared, Dataset: df_5m, Mean lag: 6.170068027210885, Median lag: 1.0
Processing df_exp_2.parquet...
Folder: minidc_prepared, Dataset: df_exp_2, Mean lag: 8.076923076923077, Median lag: 1.0
Processing df_exp_3.parquet...
Folder: minidc_prepared, Dataset: df_exp_3, Mean lag: 7.769230769230769, Median lag: 1.0
Processing df_exp_1.parquet...
Folder: minidc_prepared, Dataset: df_exp_1, Mean lag: 7.3076923076923075, Median lag: 1.0
Processing df_exp_5.parquet...
Folder: minidc_prepared, Dataset: df_exp_5, Mean lag: 5.485207100591716, Median lag: 1.0
Processing df_exp_4.parquet...
Folder: minidc_pre

In [7]:
results_df

,folder,dataset,median_lag
0,dc1_prepared,df_rack_1,5.0
1,dc1_prepared,df_rack_3,5.0
2,dc1_prepared,df_rack_2,7.0
3,dc2_2_prepared,df_5m,1.0
4,minidc_prepared,df_exp_2,1.0
5,minidc_prepared,df_exp_3,1.0
6,minidc_prepared,df_exp_1,1.0
7,minidc_prepared,df_exp_5,1.0
8,minidc_prepared,df_exp_4,3.0
9,simulation_prepared,df_sim_4,5.0


In [8]:
results_df['median_lag'].mean()

4.333333333333333

The median lag of the maximun correlation in all targets is about 5. This is selected as the maximun lag for the ML modelling.